# [SITCOM-2111] Temperature Forecast Evaluation

* How accurate is the temperature forecast data provided by MeteoBlue?
* What is the different between forecast data and the actual data?

[SITCOM-2111]: https://ls.st/SITCOM-2111

## Setup

In [ ]:
day_obs = 20250601
number_of_days = 14

In [ ]:
import logging
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

from matplotlib.lines import Line2D

from lsst.summit.utils.efdUtils import (
    getEfdData,
    getDayObsEndTime,
    getDayObsForTime,
    getDayObsStartTime,
    makeEfdClient,
)
from astropy.time import Time

In [ ]:
# Initialize an EFD client
efd_client = makeEfdClient()

# Constants used in the notebook
ess_weather_station_sal_index = 301
m1m3_inside_cell_sal_index = 113
dome_inside_sal_index = 111

# Set global font size for labels, titles, and ticks
plt.rcParams.update(
    {
        "axes.grid": True,
        "axes.labelsize": 12,
        "axes.titlesize": 14,
        "axes.formatter.useoffset": False,
        "axes.formatter.use_mathtext": False,
        "axes.formatter.limits": (-100, 100),
        "figure.figsize": (11, 6),
        "font.size": 12,
        "grid.color": "#b0b0b0",
        "grid.linestyle": ":",
        "grid.linewidth": 0.5,
        "grid.alpha": 0.75,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
    }
)

## Helper Functions

In [ ]:
def get_forecast_closest_to_publication(_df):
    """
    For each forecasted row, selects the temperature forecast whose timestamp is closest to the publication timestamp.

    Parameters
    ----------
    _df : pd.DataFrame
        DataFrame with columns: published_timestamp, forecast_timestamp, temperature, temperatureSpread.

    Returns
    -------
    pd.DataFrame
        DataFrame indexed by published_timestamp, containing the closest forecasted temperature and spread.
    """
    import pandas as pd

    _df = _df.copy()

    # Compute the time difference
    _df["time_diff"] = (_df["forecast_timestamp"] - _df["published_timestamp"]).abs()

    # Sort by time difference and keep the closest forecast per published_timestamp
    closest = (
        _df.sort_values("time_diff")
        .groupby("published_timestamp")
        .first()
        .drop(columns=["time_diff"])
    )

    return closest


def get_latest_forecast(_df, start_time=None, end_time=None):
    """
    From a reformatted forecast DataFrame, selects forecast rows
    where forecast_timestamp falls between two consecutive published timestamps.

    Parameters
    ----------
    _df : pd.DataFrame
        DataFrame with columns: published_timestamp, forecast_timestamp, temperature, temperatureSpread.
    start_time : datetime-like, optional
        Minimum forecast_timestamp to include.
    end_time : datetime-like, optional
        Maximum forecast_timestamp to include.

    Returns
    -------
    pd.DataFrame
        DataFrame indexed by forecast_timestamp, containing only selected forecasts.
    """
    import pandas as pd
    from astropy.time import Time

    if isinstance(start_time, Time):
        start_time = start_time.to_datetime().replace(tzinfo=None)
    if isinstance(end_time, Time):
        end_time = end_time.to_datetime().replace(tzinfo=None)

    _df = _df.copy()
    _df["forecast_timestamp"] = pd.to_datetime(_df["forecast_timestamp"])
    _df["published_timestamp"] = pd.to_datetime(_df["published_timestamp"])

    _df = _df.sort_values("published_timestamp")

    # Container for filtered rows
    filtered = []

    # Group forecasts between each pair of published timestamps
    for i in range(len(_df.published_timestamp.unique()) - 1):
        t0 = _df.published_timestamp.unique()[i]
        t1 = _df.published_timestamp.unique()[i + 1]
        mask = (
            (_df["forecast_timestamp"] >= t0)
            & (_df["forecast_timestamp"] < t1)
            & (_df["published_timestamp"] == t0)
        )
        filtered.append(_df[mask])

    result_df = pd.concat(filtered, ignore_index=True)

    # Filter by time range if specified
    if start_time is not None:
        result_df = result_df[
            result_df["forecast_timestamp"] >= pd.to_datetime(start_time)
        ]
    if end_time is not None:
        result_df = result_df[
            result_df["forecast_timestamp"] <= pd.to_datetime(end_time)
        ]

    # Set index and return
    result_df = result_df.set_index("forecast_timestamp")
    return result_df.sort_index()


def plot_hourly_weather_forecast(df, title="hourly_weather_forecast"):
    """
    Query the temperatue forecasts for a current day.
    """
    fig, ax = plt.subplots(num=title, ncols=1, nrows=1)

    ax.plot(df.timestamp, df.temperature)
    ax.set_xlabel("Time [UTC]")
    ax.set_ylabel("Temperature (ºC)")
    fig.autofmt_xdate()
    fig.tight_layout()

    plt.savefig(f"{title}.png")
    plt.show()


def query_daily_weather_forecast(client, day_obs_start, day_obs_end):
    """
    Query the temperatue forecasts for a current day.
    """
    start_time = getDayObsStartTime(day_obs_start)
    end_time = getDayObsEndTime(day_obs_end)

    _df = getEfdData(
        client=client,
        topic="lsst.sal.WeatherForecast.dailyTrend",
        columns=[
            "timestamp0",
            "temperatureMin0",
            "temperatureMean0",
            "temperatureMax0",
        ],
        begin=start_time,
        end=end_time,
    )

    _df["timestamp0"] = pd.to_datetime(
        _df["timestamp0"], unit="s", utc=True
    ).dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")

    return _df


def query_ess_inside_dome(client, start_time, end_time):
    """
    Query the actual temperature inside the dome.
    """
    if not isinstance(start_time, Time):
        start_time = Time(start_time.to_pydatetime(), scale="utc")
    if not isinstance(end_time, Time):
        end_time = Time(end_time.to_pydatetime(), scale="utc")

    _df = getEfdData(
        client=client,
        topic="lsst.sal.ESS.temperature",
        columns=["temperatureItem0", "salIndex", "location"],
        begin=start_time,
        end=end_time,
    )

    # Select the data from the weather station using the salIndex
    mask = _df.salIndex == dome_inside_sal_index
    _df = _df[mask]

    # We do not need the salIndex anymore
    _df = _df.drop(columns=["salIndex"])

    # Get the rolling min/mean/max values for the temperature
    _df = _df.rename(columns={"temperatureItem0": "temperature"})
    _df = _df.resample("1min").agg({"temperature": ["min", "mean", "max"]})
    _df.columns = _df.columns.droplevel(0)

    return _df


def query_ess_weather_station(client, start_time, end_time):
    """
    Query the actual temperature from the weather station.
    """
    if not isinstance(start_time, Time):
        start_time = Time(start_time.to_pydatetime(), scale="utc")
    if not isinstance(end_time, Time):
        end_time = Time(end_time.to_pydatetime(), scale="utc")

    _df = getEfdData(
        client=client,
        topic="lsst.sal.ESS.temperature",
        columns=["temperatureItem0", "salIndex", "location"],
        begin=start_time,
        end=end_time,
    )

    # Select the data from the weather station using the salIndex
    mask = _df.salIndex == ess_weather_station_sal_index
    _df = _df[mask]

    # Get the rolling min/mean/max values for the temperature
    _df = _df.rename(columns={"temperatureItem0": "temperature"})
    _df = _df.resample("1min").agg({"temperature": ["min", "mean", "max"]})
    _df.columns = _df.columns.droplevel(0)

    return _df


def query_hourly_weather_forecast(client, day_obs_start, day_obs_end=None):
    """
    Query the forecast for a given day
    """
    if day_obs_end is None:
        day_obs_end = day_obs_start

    start_time = getDayObsStartTime(day_obs_start)
    end_time = getDayObsEndTime(day_obs_end)

    NPOINTS = 336

    _df = getEfdData(
        client=client,
        topic="lsst.sal.WeatherForecast.hourlyTrend",
        columns=[f"timestamp{i}" for i in range(NPOINTS)]
        + [f"temperature{i}" for i in range(NPOINTS)]
        + [f"temperatureSpread{i}" for i in range(NPOINTS)],
        begin=start_time,
        end=end_time,
    )

    for i in range(NPOINTS):
        _df[f"timestamp{i}"] = pd.to_datetime(
            _df[f"timestamp{i}"], unit="s", utc=True
        ).dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")

    return _df


def unpack_hourly_weather_forecast(s):
    """
    Convert a Pandas Series into a DataFrame.

    Parameters
    ----------
    s : pd.Series
        A series containing timestampX, temperatureX,
        and temperatureSpreadX where X is a number from 0 to 335.
    """
    # Clean and reshape
    _df = s.rename_axis("key").reset_index(name="value")
    _df["field"] = _df["key"].str.extract(r"([a-zA-Z]+(?:Spread)?)")[0]
    _df["index"] = _df["key"].str.extract(r"(\d+)$")[0].astype("Int64")

    # Pivot to final table
    _df = _df.pivot(index="index", columns="field", values="value").reset_index(
        drop=True
    )
    _df["timestamp"] = pd.to_datetime(_df["timestamp"])
    _df = _df.where(pd.notnull(_df), pd.NA)

    mask = _df.timestamp < (_df.timestamp[0] + pd.Timedelta(hours=48))
    _df = _df[mask]

    return _df


def unpack_hourly_weather_forecast_dataframe(df):
    """
    Reformats a temperature forecast DataFrame into long format.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with index as publication time and columns temperatureX and temperatureSpreadX.

    Returns
    -------
    pd.DataFrame
        Reformatted DataFrame with columns:
        - published_timestamp
        - forecast_timestamp
        - temperature
        - temperatureSpread
    """
    records = []

    for published_time, row in df.iterrows():
        for i in range(336):
            temp = row.get(f"temperature{i}", None)
            spread = row.get(f"temperatureSpread{i}", None)
            forecast_time = published_time + pd.Timedelta(hours=i)
            records.append(
                {
                    "published_timestamp": published_time,
                    "forecast_timestamp": forecast_time,
                    "temperature": temp,
                    "temperatureSpread": spread,
                }
            )

    return pd.DataFrame(records)

## Analysis

### Hourly Forecast

The telemetry comes packed with 24 * 14 columns (336) for each telemetry value 
and each time we get new published data.  
Columns ending with 0 correspond to the forecast for the first hour after the publication time.  
Columns ending with 1 correspond to the forecast fot the second hour after the publication time.  
And so it goes up to 14 days.  

The forecast is published twice a day.   
Once near 4 UTC and once near 16h UTC.

In [ ]:
df_forecast = unpack_hourly_weather_forecast(
    query_hourly_weather_forecast(efd_client, day_obs).iloc[0]
)

df_weather_station = query_ess_weather_station(
    efd_client, df_forecast.timestamp.iloc[0], df_forecast.timestamp.iloc[-1]
)

df_inside_dome = query_ess_inside_dome(
    efd_client, df_forecast.timestamp.iloc[0], df_forecast.timestamp.iloc[-1]
)

In [ ]:
print("MeteoBlue hourly data-frame - head")
df_forecast.head()

In [ ]:
print("Weather station data-frame - head")
df_weather_station.head()

In [ ]:
print("Inside Dome data-frame - head")
df_inside_dome.head()

In [ ]:
title = f"MeteoBlue Reliability Study - {day_obs}"

fig, ax = plt.subplots(num=title, ncols=1, nrows=1)

ax.plot(df_forecast.timestamp, df_forecast.temperature, label="MeteoBlue Hourly Trend")
ax.plot(
    df_weather_station.index,
    df_weather_station["mean"],
    label="Weather Station Temperature",
)
ax.plot(df_inside_dome.index, df_inside_dome["mean"], label="Inside Dome Temperature")

ax.set_xlabel("Time [UTC]")
ax.set_ylabel("Temperature (ºC)")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))

fig.suptitle(title)
fig.autofmt_xdate()
fig.tight_layout()

plt.savefig(f"{title}.png")
plt.show()

## Hourly Forecast for multiple days

In [ ]:
time_start = getDayObsStartTime(day_obs) - pd.Timedelta(number_of_days, unit="days")
time_end = getDayObsEndTime(day_obs)

day_obs_start = getDayObsForTime(time_start)
day_obs_end = day_obs
print(f"Query data from {day_obs_start} to {day_obs_end}")

df_forecast = get_latest_forecast(
    unpack_hourly_weather_forecast_dataframe(
        query_hourly_weather_forecast(efd_client, day_obs_start, day_obs_end)
    )
)

df_weather_station = query_ess_weather_station(efd_client, time_start, time_end)

df_inside_dome = query_ess_inside_dome(efd_client, time_start, time_end)

In [ ]:
title = f"Hourly Forecast from {day_obs_start} to {day_obs_end}"
fig, ax = plt.subplots(num=title)

# Plot forecast
ax.plot(df_forecast["temperature"], label="Meteoblue Latests Forecast")

# Plot real data
ax.plot(df_weather_station["mean"], label="Weather Station Temperature")
ax.plot(df_inside_dome["mean"], label="Inside Dome Temperature")

# Plot every time the data was published
for pub_t in df_forecast["published_timestamp"].unique():
    ax.axvline(x=pub_t, ls=":", alpha=0.5)

# Add it to the existing legend
legend_line = Line2D(
    [0], [0], color="gray", linestyle="--", label="Published Timestamp"
)

handles, labels = ax.get_legend_handles_labels()
handles.append(legend_line)
labels.append("Published Timestamp")

ax.legend(handles, labels, ncols=2)

# Cosmetics
t_start = getDayObsStartTime(day_obs_start).to_datetime()
t_end = getDayObsEndTime(day_obs_end).to_datetime()

ax.set_xlim(t_start, t_end)
ax.set_xlabel("Time [UTC]")
ax.set_ylabel("Temperature [deg C]")

fig.suptitle(title)
fig.tight_layout()
fig.autofmt_xdate()

plt.savefig(f"hourly_forecast_{day_obs_start}_to_{day_obs_end}.png")
plt.show()